# Doodle Alive: free LTX-Video GPU server (Kaggle or Colab)

Runs the same LTX-Video model as the Hugging Face Space on a **free notebook GPU**, behind a public
`https://….gradio.live` link with the **same API** (`/image_to_video`). The Doodle Alive backend uses it
automatically when every Hugging Face key is out of quota.

## Kaggle (recommended: about 30 free GPU hours a week)
1. kaggle.com → **Create → New Notebook → File → Import Notebook** → upload this file.
2. Right panel → **Settings**: *Accelerator* = **GPU T4 x2**, *Internet* = **On** (needs a phone-verified account).
3. To keep it running up to 12 h with the tab closed: **Save Version → Save & Run All (Commit)**, then open the
   version's **Logs** and copy the `https://….gradio.live` link. (Or just **Run All** while you keep the tab open.)
4. Paste the link in the Doodle Alive admin panel (page URL + `#admin`) → *Free GPU servers* → **Kaggle** → Save.

## Colab (backup)
File → Upload notebook → *Runtime → Change runtime type* → **T4 GPU** → *Run all* → paste the link under **Colab**.
Keep the tab open; free Colab disconnects when idle.

First start downloads the model (~10 GB, a few minutes). One video then takes roughly 1–3 minutes on a T4.

In [ ]:
%pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg gradio

In [ ]:
import random
import tempfile

import torch
from diffusers import LTXConditionPipeline
from diffusers.utils import export_to_video
from PIL import Image

assert torch.cuda.is_available(), 'No GPU: turn on the GPU accelerator in the notebook settings'
print('GPU:', torch.cuda.get_device_name(0))

# 2B model: fits a free 16 GB GPU with CPU offload (the 13B 0.9.7/0.9.8 models do not).
pipe = LTXConditionPipeline.from_pretrained('Lightricks/LTX-Video-0.9.5', dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()  # keeps only the part that is running on the GPU
pipe.vae.enable_tiling()
print('Model ready')

In [ ]:
NEGATIVE = 'worst quality, inconsistent motion, blurry, jittery, distorted'
FPS = 24
MAX_SIDE = 640  # smaller = faster; must end up a multiple of 32


def _size(image):
    scale = MAX_SIDE / max(image.size)
    return [max(32, round(side * scale / 32) * 32) for side in image.size]


# Same name and parameters as the Lightricks/ltx-video-distilled Space, so the backend calls both the same way.
def image_to_video(prompt, input_image_filepath, mode='image-to-video', duration_ui=2):
    image = Image.open(input_image_filepath).convert('RGB')
    width, height = _size(image)
    num_frames = int(float(duration_ui) * FPS) // 8 * 8 + 1  # LTX wants 8k+1 frames
    seed = random.randint(0, 2**31 - 1)
    frames = pipe(
        prompt=prompt or 'The drawing comes to life with gentle playful motion, static camera',
        negative_prompt=NEGATIVE,
        image=image.resize((width, height)),
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=30,
        guidance_scale=3.0,
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        generator=torch.Generator().manual_seed(seed),
    ).frames[0]
    out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    export_to_video(frames, out, fps=FPS)
    return out, seed

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=image_to_video,
    inputs=[
        gr.Textbox(label='prompt'),
        gr.Image(type='filepath', label='input_image_filepath'),
        gr.Textbox(value='image-to-video', label='mode'),
        gr.Number(value=2, label='duration_ui'),
    ],
    outputs=[gr.Video(label='Generated Video'), gr.Number(label='Seed')],
    api_name='image_to_video',
    title='Doodle Alive GPU server (LTX-Video)',
)
demo.queue(max_size=20)
_, _, share_url = demo.launch(share=True, prevent_thread_lock=True)
print('=' * 70)
print('PASTE THIS LINK in the Doodle Alive admin panel (page URL + #admin):')
print(share_url)
print('=' * 70)
demo.block_thread()  # keeps the server running (needed for Kaggle 'Save & Run All')